# OUSIA Phase 1 — Gemma-4-E4B-it Fine-Tune
**Neo-Humanism Research Flagship** | Colab Enterprise (A100 40GB)

LoRA: r=16, seq=1024, batch=4, grad_accum=4 → effective batch 16
Est. runtime: 2–3 hours on A100

## 1. Setup

In [ ]:
# Clone repo (if not already mounted via GitHub integration)
!git clone https://github.com/plntrprotocol/aureth-training.git /content/aureth-training 2>/dev/null || echo "Repo already mounted"
%cd /content/aureth-training

In [ ]:
# Install dependencies
!pip install -q --upgrade pip
!pip install -q transformers peft datasets accelerate bitsandbytes torch trl huggingface_hub
!pip install -q tensorboard google-cloud-storage

In [ ]:
# Login to HuggingFace (token stored in Colab Secrets as HF_TOKEN)
from google.colab import userdata
import os

hf_token = userdata.get('HF_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print(f"HF_TOKEN set ({len(hf_token)} chars)")
else:
    raise RuntimeError("HF_TOKEN not found in Colab Secrets. Add it via ⚙️ → Secrets.")

## 2. Verify Gemma 4 E4B Pulls

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/gemma-4-E4B-it"

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
print(f"Tokenizer loaded: {tokenizer.vocab_size} tokens")

print(f"\nLoading model: {MODEL_ID} (QLoRA 4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    device_map="cuda",
    load_in_4bit=True,
    torch_dtype="bfloat16",
)
print(f"Model loaded: {model.num_parameters():,} parameters")

## 3. Load Datasets

In [ ]:
from datasets import load_dataset, interleave_datasets

print("Loading Hermes-3-Dataset...")
hermes_ds = load_dataset("NousResearch/Hermes-3-Dataset", split="train", streaming=True)
print(f"Hermes-3: {hermes_ds.dataset_size if hasattr(hermes_ds, 'dataset_size') else 'streaming'}")

print("Loading function calling dataset...")
fc_ds = load_dataset("NousResearch/hermes-function-calling-v1", split="train", streaming=True)

print("Loading refusal dataset...")
refusal_ds = load_dataset("NousResearch/RefusalDataset", split="train", streaming=False)

print(f"\nAll datasets loaded. Hermes-3: {fc_ds.n_shards} shards, RefusalDataset: {len(refusal_ds)} examples")

## 4. Apply LoRA Adapter

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

print("Applying LoRA adapter...")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Training

In [ ]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer
import os

OUTPUT_DIR = "/content/output/aureth-phase1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=0.5,
    logging_steps=50,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    save_total_limit=3,
    bf16=True,
    optim="paged_adamw_8bit",
    dataloader_num_workers=4,
    report_to=["tensorboard"],
    remove_unused_columns=False,
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=hermes_ds,
    data_collator=data_collator,
    tokenizer=tokenizer,
    max_seq_length=1024,
)

print("Starting training...")
trainer.train()

## 6. Save Adapter

In [ ]:
# Save LoRA adapter
adapter_path = "/content/aureth-phase1-adapter"
trainer.save_model(adapter_path)
print(f"Adapter saved to: {adapter_path}")

# Zip for download
!cd /content && zip -r aureth-phase1-adapter.zip aureth-phase1-adapter/
print("Zip created: aureth-phase1-adapter.zip")